# WifiSignalWave: build Android APK at zero cost
This notebook runs the **real native Android build** in your own Google Colab session, independent of GitHub Actions runners. Click **Runtime → Run all**, follow Colab prompts, and the resulting `app-debug.apk` will be offered as a download **only if tests and compilation complete**.

This downloads official Android SDK/Gradle dependencies to the temporary Colab runtime. It uses **no release keys**, **does not modify GitHub**, and **does not send any Wi-Fi measurements**. Free Colab usage and availability are subject to Google's limits. Do not use a paid Colab runtime.


In [ ]:
# WifiSignalWave: free Google Colab debug APK build
# Executes only inside the Colab runtime after YOU press Run.
from pathlib import Path
import os, shutil, subprocess, urllib.request, zipfile, hashlib

def run(args, cwd=None, env=None):
    print(">", " ".join(map(str, args)), flush=True)
    subprocess.run(list(map(str, args)), cwd=cwd, env=env, check=True)

root = Path("/content/WifiSignalWave")
repo = "https://github.com/Spiritjcrow/WifiSignalWave.git"
branch = "feature/wifisignalwave-android-v1-1-20261121"
if root.exists():
    run(["git", "-C", root, "fetch", "origin", branch])
    run(["git", "-C", root, "checkout", "-f", branch])
    run(["git", "-C", root, "reset", "--hard", "FETCH_HEAD"])
else:
    run(["git", "clone", "--depth", "1", "--branch", branch, repo, root])

# Android Gradle Plugin 8.2.x requires JDK 17. Pin the runtime to JDK 17.
jdk = Path("/usr/lib/jvm/java-17-openjdk-amd64")
if not (jdk / "bin/java").exists():
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"])
if not (jdk / "bin/java").exists():
    raise RuntimeError("JDK 17 unavailable in this runtime")
env = os.environ.copy()
env["JAVA_HOME"] = str(jdk)
env["PATH"] = str(jdk / "bin") + os.pathsep + env.get("PATH", "")
run(["java", "-version"], env=env)

# Download official Android command-line tools and install the Android 34 SDK.
sdk = Path("/content/android-sdk")
env["ANDROID_HOME"] = str(sdk)
env["ANDROID_SDK_ROOT"] = str(sdk)
tool = sdk / "cmdline-tools/latest/bin/sdkmanager"
if not tool.exists():
    sdk.mkdir(parents=True, exist_ok=True)
    tmp = Path("/content/wifisignal-sdk-extract")
    tmp.mkdir(parents=True, exist_ok=True)
    archive = Path("/content/android-commandlinetools.zip")
    url = "https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip"
    print("Downloading Android SDK command-line tools from Google...", flush=True)
    urllib.request.urlretrieve(url, archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(tmp)
    (sdk / "cmdline-tools").mkdir(parents=True, exist_ok=True)
    shutil.move(str(tmp / "cmdline-tools"), str(sdk / "cmdline-tools/latest"))
    archive.unlink()
run(["bash", "-c", "yes | \"$1\" --sdk_root=\"$2\" --licenses >/content/wifisignal-licenses.log", "bash", tool, sdk], env=env)
run([tool, "--sdk_root=" + str(sdk), "platforms;android-34", "build-tools;34.0.0", "platform-tools"], env=env)

# Execute the project's offline tests, then JUnit + actual APK compilation.
run(["bash", "tools/check.sh"], cwd=root, env=env)
run(["bash", "tools/build-debug-apk.sh"], cwd=root, env=env)
apk = root / "app/build/outputs/apk/debug/app-debug.apk"
if not apk.is_file() or apk.stat().st_size <= 0:
    raise RuntimeError("APK build did not produce app-debug.apk")
digest = hashlib.sha256(apk.read_bytes()).hexdigest()
print("\nSUCCESS: compiled debug APK", apk)
print("Size:", apk.stat().st_size, "bytes")
print("SHA-256:", digest)
from google.colab import files
files.download(str(apk))
